# Strategy probability opportunity calibration — strict 42-fold audit

## TL;DR

The pre-edge opportunity cohort contains **10,609 observations** across **631 terminal conditions** and 42 chronological folds. The strictly prior-fold calibrator is **rejected before strategy integration**. Its condition-weighted Brier improvement over the executable market is `-0.000145` and log-loss improvement is `+0.000017`. No promotion gate is changed by this diagnostic; exact executable replay remains authoritative.

## Context and methods

The prior calibration experiment scored only trades selected by the old threshold. That is not a valid counterfactual because changing a probability changes which trades exist. This notebook instead uses the first strategy candidate per condition and UTC second after all non-edge gates and before EV, stale-edge, and minimum-edge checks.

Each terminal condition receives total weight one, regardless of how many seconds it contributes. Folds stay chronological. For every scored fold, model family and ridge penalty are selected on the final two strictly prior folds, then refit on all prior folds. The calibrator may choose executable-market identity, a market intercept, or a bounded shrinkage from market log-odds toward model log-odds. No scored-fold labels enter fitting.

Calibration is eligible for exact strategy replay only if it beats the market on condition-weighted Brier score and log loss overall and in both chronological halves, the 95% condition bootstrap lower bounds are positive for both metrics, and support remains at least 30 scored folds and 300 conditions. This is a mechanism screen, not a profitability or promotion result.

In [1]:
from __future__ import annotations
import collections
import datetime as dt
import glob
import json
import math
import re
from pathlib import Path
import numpy as np

DATA_DIR = Path('/private/tmp/polymomentum_strategy_a_plus_20260715/calibration_opportunities_strict42_latency202')
SUMMARY_JSON = Path('/Users/ttoomm/Documents/PolyMomentum/deploy/promotions/evidence/strategy_registry/20260715_probability_opportunity_calibration_strict42.json')
EXPECTED_LATENCY_MS = 202
MIN_PRIOR_FOLDS = 8
VALIDATION_FOLDS = 2
RIDGES = (0.0, 0.01, 0.1, 1.0, 10.0)
EPS = 1e-6


## Data and quality controls

In [2]:
files = sorted(DATA_DIR.glob('fold_*_opportunities.json'))
assert len(files) == 42, f'expected 42 opportunity exports, found {len(files)}'
rows = []
fold_metadata = []
for path in files:
    match = re.match(r'fold_(\d+)_', path.name)
    assert match, path
    fold = int(match.group(1))
    payload = json.loads(path.read_text())
    report_path = path.with_name(path.name.replace('_opportunities.json', '_report.json'))
    report = json.loads(report_path.read_text())
    variant = report['variants'][0]
    assert payload['continuous'] is True
    assert payload['latency_ms'] == EXPECTED_LATENCY_MS
    assert payload['variant_count'] == 1
    assert payload['data_manifest']['complete'] is True
    assert len(payload['rows']) == payload['row_count']
    assert variant['trades'] == 0 and variant['execution_attempts'] == 0
    fold_metadata.append({
        'fold': fold, 'rows': payload['row_count'],
        'conditions': payload['condition_count'],
        'manifest_hash': payload['data_manifest']['manifest_hash'],
    })
    for wrapped in payload['rows']:
        opportunity = dict(wrapped['opportunity'])
        decision = opportunity.pop('decision')
        opportunity.update({
            'fold': fold,
            'fair_value': decision['fair_value'],
            'market_price': decision['market_price'],
            'minutes_remaining': decision['minutes_remaining'],
            'z_score': decision['z_score'],
            'direction': decision['direction'],
            'zone': decision['zone'],
        })
        rows.append(opportunity)

folds = sorted({row['fold'] for row in rows})
keys = [(row['condition_id'], row['sampling_second']) for row in rows]
condition_rows = collections.Counter(row['condition_id'] for row in rows)
directions_by_condition = collections.defaultdict(set)
folds_by_condition = collections.defaultdict(set)
for row in rows:
    directions_by_condition[row['condition_id']].add(row['actual_direction'])

    folds_by_condition[row['condition_id']].add(row['fold'])

quality = {
    'files': len(files),
    'folds': len(folds),
    'rows': len(rows),
    'conditions': len(condition_rows),
    'duplicate_condition_seconds': len(keys) - len(set(keys)),
    'condition_direction_conflicts': sum(len(v) != 1 for v in directions_by_condition.values()),
    'conditions_crossing_folds': sum(len(v) != 1 for v in folds_by_condition.values()),
    'outcome_mapping_mismatches': sum(row['won'] != (row['direction'] == row['actual_direction']) for row in rows),
    'sampling_floor_mismatches': sum(math.floor(row['decision_timestamp_s']) != row['sampling_second'] for row in rows),
    'invalid_probabilities': sum(not (0.0 <= row['fair_value'] <= 1.0 and 0.0 <= row['market_price'] <= 1.0) for row in rows),
    'incomplete_manifests': 0,
    'nonzero_trade_reports': 0,
    'evaluation_results': dict(collections.Counter(row['evaluation_result'] for row in rows)),
    'resolution_sources': dict(collections.Counter(row['resolution_source'] for row in rows)),
    'rows_per_condition': {
        'min': min(condition_rows.values()),
        'median': float(np.median(list(condition_rows.values()))),
        'max': max(condition_rows.values()),
    },
}
assert quality['folds'] == 42
assert quality['duplicate_condition_seconds'] == 0
assert quality['condition_direction_conflicts'] == 0
assert quality['conditions_crossing_folds'] == 0
assert quality['outcome_mapping_mismatches'] == 0
assert quality['sampling_floor_mismatches'] == 0
assert quality['invalid_probabilities'] == 0
quality


{'files': 42,
 'folds': 42,
 'rows': 10609,
 'conditions': 631,
 'duplicate_condition_seconds': 0,
 'condition_direction_conflicts': 0,
 'conditions_crossing_folds': 0,
 'outcome_mapping_mismatches': 0,
 'sampling_floor_mismatches': 0,
 'invalid_probabilities': 0,
 'incomplete_manifests': 0,
 'nonzero_trade_reports': 0,
 'evaluation_results': {'low_edge': 10504, 'edge_too_high_stale': 105},
 'resolution_sources': {'polymarket_terminal': 10609},
 'rows_per_condition': {'min': 1, 'median': 11.0, 'max': 60}}

The export is suitable for calibration diagnosis if all assertions above pass. Repeated seconds are not independent outcomes; the weighting below prevents dense conditions from dominating the fit or score.

## Causal calibration model

In [3]:
def clip_probability(p):
    return np.clip(np.asarray(p, dtype=float), EPS, 1.0 - EPS)

def logit(p):
    p = clip_probability(p)
    return np.log(p / (1.0 - p))

def sigmoid(x):
    x = np.asarray(x, dtype=float)
    out = np.empty_like(x)
    positive = x >= 0
    out[positive] = 1.0 / (1.0 + np.exp(-x[positive]))
    exp_x = np.exp(x[~positive])
    out[~positive] = exp_x / (1.0 + exp_x)
    return out

def metric_pair(y, p, w):
    p = clip_probability(p)
    w = np.asarray(w, dtype=float)
    w = w / w.sum()
    return {
        'brier': float(np.sum(w * (p - y) ** 2)),
        'log_loss': float(-np.sum(w * (y * np.log(p) + (1.0 - y) * np.log(1.0 - p)))),
    }

def design(model, market, fair):
    base = logit(market)
    delta = logit(fair) - base
    if model == 'identity':
        return base, np.empty((len(base), 0)), np.array([]), np.array([])
    if model == 'market_intercept':
        return base, np.ones((len(base), 1)), np.array([-1.5]), np.array([1.5])
    if model == 'fair_shrink':
        return base, np.column_stack([np.ones(len(base)), delta]), np.array([-1.5, 0.0]), np.array([1.5, 1.0])
    raise ValueError(model)

def penalized_objective(y, eta, w, beta, ridge):
    p = clip_probability(sigmoid(eta))
    w = w / w.sum()
    loss = -np.sum(w * (y * np.log(p) + (1.0 - y) * np.log(1.0 - p)))
    return float(loss + 0.5 * ridge * np.dot(beta, beta))

def fit_model(model, market, fair, y, w, ridge):
    base, X, lower, upper = design(model, market, fair)
    if model == 'identity':
        return np.array([])
    beta = np.zeros(X.shape[1])
    normalized_w = w / w.sum()
    for _ in range(100):
        eta = base + X @ beta
        p = sigmoid(eta)
        gradient = X.T @ (normalized_w * (p - y)) + ridge * beta
        curvature = normalized_w * p * (1.0 - p)
        hessian = X.T @ (curvature[:, None] * X) + (ridge + 1e-8) * np.eye(X.shape[1])
        step = np.linalg.solve(hessian, gradient)
        step = np.clip(np.nan_to_num(step, nan=0.0, posinf=10.0, neginf=-10.0), -10.0, 10.0)
        old = penalized_objective(y, eta, w, beta, ridge)
        scale = 1.0
        accepted = False
        while scale >= 1e-6:
            candidate = np.clip(beta - scale * step, lower, upper)
            candidate_eta = base + X @ candidate
            if penalized_objective(y, candidate_eta, w, candidate, ridge) <= old + 1e-12:
                accepted = True
                break
            scale *= 0.5
        if not accepted or np.max(np.abs(candidate - beta)) < 1e-9:
            break
        beta = candidate
    return beta

def predict_model(model, beta, market, fair):
    base, X, _, _ = design(model, market, fair)
    return sigmoid(base + X @ beta)


In [4]:
n = len(rows)
fold_array = np.array([row['fold'] for row in rows], dtype=int)
condition_array = np.array([row['condition_id'] for row in rows], dtype=object)
y = np.array([row['won'] for row in rows], dtype=float)
market = np.array([row['market_price'] for row in rows], dtype=float)
fair = np.array([row['fair_value'] for row in rows], dtype=float)
weights = np.array([1.0 / condition_rows[row['condition_id']] for row in rows], dtype=float)
candidate_models = ('identity', 'market_intercept', 'fair_shrink')
predictions = np.full(n, np.nan)
fold_decisions = []

for position in range(MIN_PRIOR_FOLDS, len(folds)):
    scored_fold = folds[position]
    prior_folds = folds[:position]
    validation_folds = prior_folds[-VALIDATION_FOLDS:]
    fit_folds = prior_folds[:-VALIDATION_FOLDS]
    fit_mask = np.isin(fold_array, fit_folds)
    validation_mask = np.isin(fold_array, validation_folds)
    choices = []
    for model in candidate_models:
        ridges = (0.0,) if model == 'identity' else RIDGES
        for ridge in ridges:
            beta = fit_model(model, market[fit_mask], fair[fit_mask], y[fit_mask], weights[fit_mask], ridge)
            p = predict_model(model, beta, market[validation_mask], fair[validation_mask])
            score = metric_pair(y[validation_mask], p, weights[validation_mask])
            choices.append((score['log_loss'], score['brier'], len(beta), model, ridge))
    _, _, _, selected_model, selected_ridge = min(choices)
    prior_mask = np.isin(fold_array, prior_folds)
    beta = fit_model(selected_model, market[prior_mask], fair[prior_mask], y[prior_mask], weights[prior_mask], selected_ridge)
    scored_mask = fold_array == scored_fold
    predictions[scored_mask] = predict_model(selected_model, beta, market[scored_mask], fair[scored_mask])
    fold_decisions.append({
        'fold': scored_fold, 'prior_folds': len(prior_folds),
        'validation_folds': validation_folds, 'model': selected_model,
        'ridge': selected_ridge, 'coefficients': beta.tolist(),
        'conditions': len(set(condition_array[scored_mask])),
    })

scored = np.isfinite(predictions)
assert scored.sum() > 0
selected_model_counts = dict(collections.Counter(item['model'] for item in fold_decisions))
selected_model_counts, fold_decisions[:3]


({'identity': 12, 'fair_shrink': 16, 'market_intercept': 6},
 [{'fold': 9,
   'prior_folds': 8,
   'validation_folds': [7, 8],
   'model': 'identity',
   'ridge': 0.0,
   'coefficients': [],
   'conditions': 19},
  {'fold': 10,
   'prior_folds': 9,
   'validation_folds': [8, 9],
   'model': 'fair_shrink',
   'ridge': 10.0,
   'coefficients': [0.0011237284624350788, 0.004744719312247299],
   'conditions': 22},
  {'fold': 11,
   'prior_folds': 10,
   'validation_folds': [9, 10],
   'model': 'identity',
   'ridge': 0.0,
   'coefficients': [],
   'conditions': 16}])

## Results

In [5]:
def comparison(mask):
    market_score = metric_pair(y[mask], market[mask], weights[mask])
    fair_score = metric_pair(y[mask], fair[mask], weights[mask])
    calibrated_score = metric_pair(y[mask], predictions[mask], weights[mask])
    return {
        'conditions': len(set(condition_array[mask])),
        'rows': int(mask.sum()),
        'market': market_score, 'raw_fair': fair_score, 'calibrated': calibrated_score,
        'market_minus_raw_fair_brier': market_score['brier'] - fair_score['brier'],
        'market_minus_raw_fair_log_loss': market_score['log_loss'] - fair_score['log_loss'],
        'market_minus_calibrated_brier': market_score['brier'] - calibrated_score['brier'],
        'market_minus_calibrated_log_loss': market_score['log_loss'] - calibrated_score['log_loss'],
    }

scored_folds = [item['fold'] for item in fold_decisions]
half = len(scored_folds) // 2
first_half = scored & np.isin(fold_array, scored_folds[:half])
second_half = scored & np.isin(fold_array, scored_folds[half:])
overall = comparison(scored)
halves = {'first': comparison(first_half), 'second': comparison(second_half)}
overall, halves


({'conditions': 498,
  'rows': 7774,
  'market': {'brier': 0.14273255282818378, 'log_loss': 0.4573943788558661},
  'raw_fair': {'brier': 0.14807646110186165, 'log_loss': 0.47475294455623507},
  'calibrated': {'brier': 0.14287709806415833,
   'log_loss': 0.45737732833275346},
  'market_minus_raw_fair_brier': -0.0053439082736778665,
  'market_minus_raw_fair_log_loss': -0.017358565700368978,
  'market_minus_calibrated_brier': -0.00014454523597454139,
  'market_minus_calibrated_log_loss': 1.7050523112627225e-05},
 {'first': {'conditions': 246,
   'rows': 3797,
   'market': {'brier': 0.1363619630206016, 'log_loss': 0.44162193608673866},
   'raw_fair': {'brier': 0.14444805829717255, 'log_loss': 0.4633397269245773},
   'calibrated': {'brier': 0.13785749652076867,
    'log_loss': 0.44503887947097454},
   'market_minus_raw_fair_brier': -0.008086095276570948,
   'market_minus_raw_fair_log_loss': -0.021717790837838646,
   'market_minus_calibrated_brier': -0.0014955335001670678,
   'market_minus_c

In [6]:
condition_differences = collections.defaultdict(lambda: {'brier': [], 'log_loss': []})
for idx in np.flatnonzero(scored):
    market_p = clip_probability(market[idx])
    calibrated_p = clip_probability(predictions[idx])
    outcome = y[idx]
    condition_differences[condition_array[idx]]['brier'].append((market_p - outcome) ** 2 - (calibrated_p - outcome) ** 2)
    market_ll = -(outcome * math.log(market_p) + (1.0 - outcome) * math.log(1.0 - market_p))
    calibrated_ll = -(outcome * math.log(calibrated_p) + (1.0 - outcome) * math.log(1.0 - calibrated_p))
    condition_differences[condition_array[idx]]['log_loss'].append(market_ll - calibrated_ll)
condition_ids = sorted(condition_differences)
condition_brier = np.array([np.mean(condition_differences[c]['brier']) for c in condition_ids])
condition_log_loss = np.array([np.mean(condition_differences[c]['log_loss']) for c in condition_ids])
rng = np.random.default_rng(20260715)
bootstrap_brier = np.empty(2000)
bootstrap_log_loss = np.empty(2000)
for index in range(2000):
    draw = rng.integers(0, len(condition_ids), len(condition_ids))
    bootstrap_brier[index] = np.mean(condition_brier[draw])
    bootstrap_log_loss[index] = np.mean(condition_log_loss[draw])
bootstrap = {
    'market_minus_calibrated_brier_95pct': np.quantile(bootstrap_brier, [0.025, 0.975]).tolist(),
    'market_minus_calibrated_log_loss_95pct': np.quantile(bootstrap_log_loss, [0.025, 0.975]).tolist(),
}
bootstrap


{'market_minus_calibrated_brier_95pct': [-0.002039189044594095,
  0.0016996963068833775],
 'market_minus_calibrated_log_loss_95pct': [-0.005735544869003813,
  0.005620249899879944]}

In [7]:
fold_scores = []
for fold in scored_folds:
    mask = scored & (fold_array == fold)
    item = comparison(mask)
    item['fold'] = fold
    fold_scores.append(item)

def bucket_market(value):
    if value < 0.60: return 'lt_0.60'
    if value < 0.75: return '0.60_0.75'
    if value < 0.90: return '0.75_0.90'
    return 'gte_0.90'

segment_values = {
    'direction': np.array([row['direction'] for row in rows], dtype=object),
    'market_price': np.array([bucket_market(row['market_price']) for row in rows], dtype=object),
    'minutes_remaining': np.array([
        'lt_2' if row['minutes_remaining'] < 2 else ('2_4' if row['minutes_remaining'] < 4 else 'gte_4')
        for row in rows
    ], dtype=object),
    'observed_volatility': np.array([
        'lt_0.30' if row['observed_volatility'] < 0.30 else ('0.30_0.50' if row['observed_volatility'] < 0.50 else 'gte_0.50')
        for row in rows
    ], dtype=object),
}
segments = {}
for dimension, values in segment_values.items():
    segments[dimension] = []
    for value in sorted(set(values[scored])):
        mask = scored & (values == value)
        item = comparison(mask)
        item['value'] = value
        segments[dimension].append(item)
segments


{'direction': [{'conditions': 296,
   'rows': 4840,
   'market': {'brier': 0.12831902973118317, 'log_loss': 0.42309035335373935},
   'raw_fair': {'brier': 0.13548538757899664, 'log_loss': 0.4437729692564174},
   'calibrated': {'brier': 0.12738232323981782,
    'log_loss': 0.41993422129947783},
   'market_minus_raw_fair_brier': -0.00716635784781347,
   'market_minus_raw_fair_log_loss': -0.02068261590267806,
   'market_minus_calibrated_brier': 0.0009367064913653467,
   'market_minus_calibrated_log_loss': 0.003156132054261518,
   'value': 'down'},
  {'conditions': 203,
   'rows': 2934,
   'market': {'brier': 0.1637185549701748, 'log_loss': 0.507340831768295},
   'raw_fair': {'brier': 0.16640898772582063, 'log_loss': 0.5198596005504308},
   'calibrated': {'brier': 0.1654373961581716, 'log_loss': 0.5118942649009295},
   'market_minus_raw_fair_brier': -0.002690432755645844,
   'market_minus_raw_fair_log_loss': -0.012518768782135758,
   'market_minus_calibrated_brier': -0.0017188411879968113,

## Takeaways and decision

In [8]:
support_ok = len(scored_folds) >= 30 and overall['conditions'] >= 300
overall_ok = overall['market_minus_calibrated_brier'] > 0 and overall['market_minus_calibrated_log_loss'] > 0
halves_ok = all(
    part['market_minus_calibrated_brier'] > 0 and part['market_minus_calibrated_log_loss'] > 0
    for part in halves.values()
)
bootstrap_ok = bootstrap['market_minus_calibrated_brier_95pct'][0] > 0 and bootstrap['market_minus_calibrated_log_loss_95pct'][0] > 0
eligible = support_ok and overall_ok and halves_ok and bootstrap_ok
result = {
    'calibration_eligible': eligible,
    'support_ok': support_ok, 'overall_ok': overall_ok,
    'chronological_halves_ok': halves_ok, 'bootstrap_ok': bootstrap_ok,
    'scored_folds': len(scored_folds), 'scored_conditions': overall['conditions'],
    'market_minus_calibrated_brier': overall['market_minus_calibrated_brier'],
    'market_minus_calibrated_log_loss': overall['market_minus_calibrated_log_loss'],
    'selected_model_counts': selected_model_counts,
}
summary = {
    'schema_version': 1,
    'generated_at': dt.datetime.now(dt.timezone.utc).isoformat(),
    'methodology': {
        'sampling': 'first_pre_edge_candidate_per_condition_utc_second',
        'weighting': 'each terminal condition has total weight one',
        'selection': 'model family and ridge selected on final two strictly prior folds',
        'models': list(candidate_models), 'ridge_candidates': list(RIDGES),
        'minimum_prior_folds': MIN_PRIOR_FOLDS, 'bootstrap_seed': 20260715,
    },
    'data_quality': quality, 'result': result, 'overall': overall,
    'chronological_halves': halves, 'bootstrap': bootstrap,
    'fold_decisions': fold_decisions, 'fold_scores': fold_scores,
    'segments': segments, 'source_folds': fold_metadata,
    'promotion_posture': 'research_only_live_off',
}
SUMMARY_JSON.parent.mkdir(parents=True, exist_ok=True)
temporary = SUMMARY_JSON.with_name(SUMMARY_JSON.name + '.tmp')
temporary.write_text(json.dumps(summary, indent=2, sort_keys=True) + '\n')
temporary.replace(SUMMARY_JSON)
verdict = 'ELIGIBLE FOR EXACT REPLAY' if eligible else 'REJECTED BEFORE STRATEGY INTEGRATION'
print(verdict)
print(json.dumps(result, indent=2, sort_keys=True))


REJECTED BEFORE STRATEGY INTEGRATION
{
  "bootstrap_ok": false,
  "calibration_eligible": false,
  "chronological_halves_ok": false,
  "market_minus_calibrated_brier": -0.00014454523597454139,
  "market_minus_calibrated_log_loss": 1.7050523112627225e-05,
  "overall_ok": false,
  "scored_conditions": 498,
  "scored_folds": 34,
  "selected_model_counts": {
    "fair_shrink": 16,
    "identity": 12,
    "market_intercept": 6
  },
  "support_ok": true
}


Interpretation rules:

- `market_minus_* > 0` means the candidate improves on the executable market probability.
- A rejection means no calibration parameter should enter runtime strategy code.
- Eligibility would authorize only an exact fold-forward executable replay, not promotion.
- The unchanged A+ profitability, breadth, tail, latency, and fresh-window gates remain the final decision contract.